# Task: AI Agent for Assist water utility managers
#### in Phase 3 - Module 5: LangChain & Building Agents

An AI Agent that helps water utility managers monitor, analyze, and report on water system performance.

----------

## 1. Data Retrieval


In [1]:
import pandas as pd

df = pd.read_csv("water_meter_30days.csv")
df["DateTime"] = pd.to_datetime(df["DateTime"])

df.head(3)

,DateTime,Meter_ID,Flow_Rate(L/s),Pressure(bar),Water_Temperature(C),Water_Volume(m3)
0,2025-10-01,METER_001,1.09,2.97,16.0,3.9
1,2025-10-01,METER_002,0.87,2.92,15.7,3.1
2,2025-10-01,METER_003,1.93,3.03,14.9,7.0


### Missing Data Handling
Detect, report, and clean missing values before analysis begins.


In [2]:
REQUIRED_COLUMNS = ["DateTime", "Meter_ID", "Flow_Rate(L/s)", "Pressure(bar)", "Water_Volume(m3)"]

# Validate required columns exist
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Detect missing values
missing_vals = df[REQUIRED_COLUMNS].isnull().sum()
cols_with_missing = missing_vals[missing_vals > 0]

if not cols_with_missing.empty:
    print("Missing values detected:")
    print(cols_with_missing)
    df = df.dropna(subset=REQUIRED_COLUMNS)
    print(f"Rows after cleaning: {len(df)}")
else:
    print(f"No missing values found. Dataset ready: {len(df)} records.")


No missing values found. Dataset ready: 2160 records.


## 2. Data Analysis


### 2.1 Detect Anomalies
(e.g., leaks, unusual pressure drops).

In [3]:
def detect_anomalies(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)

df["Flow_Anomaly"]     = df.groupby("Meter_ID")["Flow_Rate(L/s)"].transform(detect_anomalies)
df["Pressure_Anomaly"] = df.groupby("Meter_ID")["Pressure(bar)"].transform(detect_anomalies)

anomaly_df = df[df["Flow_Anomaly"] | df["Pressure_Anomaly"]].copy()

print(f"Flow anomalies     : {df['Flow_Anomaly'].sum()}")
print(f"Pressure anomalies : {df['Pressure_Anomaly'].sum()}")
print(f"Total anomalies    : {len(anomaly_df)}")

Flow anomalies     : 42
Pressure anomalies : 17
Total anomalies    : 59


### 2.2 Calculate key metrics 
(total demand, leak volume, efficiency).


In [4]:
total_demand = df["Water_Volume(m3)"].sum()

estimated_leak_volume = df.loc[df["Flow_Anomaly"], "Water_Volume(m3)"].sum()

efficiency = ((total_demand - estimated_leak_volume) / total_demand * 100) if total_demand > 0 else 0.0

print(f"Total Water Demand            : {total_demand:.2f} m³")
print(f"Estimated Anomalous Flow Vol. : {estimated_leak_volume:.2f} m³")
print(f"System Efficiency             : {efficiency:.2f}%")

Total Water Demand            : 10946.00 m³
Estimated Anomalous Flow Vol. : 385.30 m³
System Efficiency             : 96.48%


## 3. Reasoning & Recommendations


In [5]:
# Determine possible causes of anomalies.

def determine_causes(row):
    causes = []
    if row["Flow_Anomaly"]:
        causes += ["Possible pipeline leak", "Unusual water demand", "Meter calibration issue"]
    if row["Pressure_Anomaly"]:
        causes += ["Possible pressure drop", "Pump issue", "Valve issue"]
    return "; ".join(causes) if causes else "No anomaly"


# Suggest corrective actions (e.g., inspect a pipeline, adjust pump schedules).

def suggest_actions(row):
    actions = []
    if row["Flow_Anomaly"]:
        actions += ["Inspect pipeline for leaks", "Check meter calibration", "Compare with nearby meters"]
    if row["Pressure_Anomaly"]:
        actions += ["Check pumps and schedules", "Inspect valves", "Review pipeline pressure"]
    return "; ".join(actions) if actions else "Continue normal monitoring"


anomaly_df["Possible_Causes"]   = anomaly_df.apply(determine_causes, axis=1)
anomaly_df["Corrective_Actions"] = anomaly_df.apply(suggest_actions, axis=1)

print("Anomaly records with causes and actions:")
anomaly_df[["Meter_ID", "Flow_Anomaly", "Pressure_Anomaly", "Possible_Causes", "Corrective_Actions"]].head()


Anomaly records with causes and actions:


,Meter_ID,Flow_Anomaly,Pressure_Anomaly,Possible_Causes,Corrective_Actions
34,METER_002,True,False,Possible pipeline leak; Unusual water demand; ...,Inspect pipeline for leaks; Check meter calibr...
61,METER_002,True,False,Possible pipeline leak; Unusual water demand; ...,Inspect pipeline for leaks; Check meter calibr...
75,METER_001,True,False,Possible pipeline leak; Unusual water demand; ...,Inspect pipeline for leaks; Check meter calibr...
77,METER_003,False,True,Possible pressure drop; Pump issue; Valve issue,Check pumps and schedules; Inspect valves; Rev...
131,METER_003,True,False,Possible pipeline leak; Unusual water demand; ...,Inspect pipeline for leaks; Check meter calibr...


## 4. Reporting

In [6]:
analysis_context = {
    "records_monitored"    : len(df),
    "meters_monitored"     : df["Meter_ID"].nunique(),
    "meter_ids"            : sorted(df["Meter_ID"].unique().tolist()),
    "total_demand"         : round(total_demand, 2),
    "estimated_leak_volume": round(estimated_leak_volume, 2),
    "efficiency"           : round(efficiency, 2),
    "total_anomalies"      : int((df["Flow_Anomaly"] | df["Pressure_Anomaly"]).sum()),
    "flow_anomalies"       : int(df["Flow_Anomaly"].sum()),
    "pressure_anomalies"   : int(df["Pressure_Anomaly"].sum()),
    "anomaly_df"           : anomaly_df,
}

for key, val in analysis_context.items():
    if key != "anomaly_df":
        print(f"  {key}: {val}")


  records_monitored: 2160
  meters_monitored: 3
  meter_ids: ['METER_001', 'METER_002', 'METER_003']
  total_demand: 10946.0
  estimated_leak_volume: 385.3
  efficiency: 96.48
  total_anomalies: 59
  flow_anomalies: 42
  pressure_anomalies: 17


## 5. Tool Integration

In [7]:
import os
import requests
from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()

# Weather API location — configurable via .env
WEATHER_LOCATION = {
    "name"     : os.getenv("WEATHER_LOCATION_NAME", "Ramallah, Palestine"),
    "latitude" : float(os.getenv("WEATHER_LATITUDE",  "31.9038")),
    "longitude": float(os.getenv("WEATHER_LONGITUDE", "35.2034")),
}
WEATHER_API_URL     = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_API_TIMEOUT = 10  # seconds


@tool
def get_water_system_analysis(query: str = "") -> str:
    """Returns water system metrics: total demand, efficiency, and anomaly counts."""
    ctx = analysis_context
    return (
        f"Records Monitored     : {ctx['records_monitored']}\n"
        f"Meters                : {', '.join(ctx['meter_ids'])}\n"
        f"Total Water Demand    : {ctx['total_demand']:.2f} m3\n"
        f"Estimated Anomalous Flow Volume : {ctx['estimated_leak_volume']:.2f} m3 (proxy estimate, not a physical measurement)\n"
        f"System Efficiency     : {ctx['efficiency']:.2f}%\n"
        f"Flow Anomalies        : {ctx['flow_anomalies']}\n"
        f"Pressure Anomalies    : {ctx['pressure_anomalies']}\n"
        f"Total Anomalies       : {ctx['total_anomalies']}"
    )


@tool
def get_anomaly_details(max_records: int = 5) -> str:
    """Returns detailed anomaly records with possible causes and corrective actions (max 20)."""
    max_records = min(int(max_records), 20)
    adf = analysis_context["anomaly_df"]
    if adf.empty:
        return "No anomalies detected in the dataset."
    lines = [f"Anomaly Details (showing {min(max_records, len(adf))} of {len(adf)}):"]
    for _, row in adf.head(max_records).iterrows():
        atype = " + ".join(
            (["Flow"] if row["Flow_Anomaly"] else []) +
            (["Pressure"] if row["Pressure_Anomaly"] else [])
        )
        lines.append(
            f"\n  [{row['DateTime']}] {row['Meter_ID']} | {atype}\n"
            f"    Causes  : {row['Possible_Causes']}\n"
            f"    Actions : {row['Corrective_Actions']}"
        )
    return "\n".join(lines)


@tool
def get_weather_data(date_str: str) -> str:
    """Fetches daily weather (temp, precipitation) for a given date (YYYY-MM-DD).
    Weather is additional context — it does not confirm the cause of any anomaly."""
    # Validate date input
    try:
        formatted_date = str(pd.to_datetime(date_str).date())
    except Exception as e:
        return f"[Weather Error] Invalid date '{date_str}'. Please use YYYY-MM-DD format. Detail: {e}"

    params = {
        "latitude": WEATHER_LOCATION["latitude"],
        "longitude": WEATHER_LOCATION["longitude"],
        "start_date": formatted_date,
        "end_date": formatted_date,
        "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
        "timezone": "auto",
    }

    try:
        r = requests.get(WEATHER_API_URL, params=params, timeout=WEATHER_API_TIMEOUT)
        r.raise_for_status()
        daily = r.json().get("daily", {})
        tmax  = daily.get("temperature_2m_max",  ["N/A"])[0]
        tmin  = daily.get("temperature_2m_min",  ["N/A"])[0]
        prec  = daily.get("precipitation_sum",   ["N/A"])[0]
        return (
            f"Weather on {formatted_date} — {WEATHER_LOCATION['name']}:\n"
            f"  Max Temp      : {tmax} °C\n"
            f"  Min Temp      : {tmin} °C\n"
            f"  Precipitation : {prec} mm\n"
            f"  Note: Weather is additional context only."
        )
    except requests.exceptions.Timeout:
        return f"[Weather Error] Request timed out for {formatted_date}."
    except Exception as e:
        return f"[Weather Error] Could not fetch weather for {formatted_date}. Detail: {e}"


@tool
def get_corrective_actions(anomaly_type: str) -> str:
    """Returns corrective actions for anomaly type: 'flow', 'pressure', or 'both'."""
    t = anomaly_type.lower().strip()
    flow_actions = [
        "Inspect the pipeline segment for possible leaks.",
        "Check and calibrate the meter.",
        "Compare readings with nearby meters.",
    ]
    pressure_actions = [
        "Check pumps and review operating schedules.",
        "Inspect valves for blockage or partial closure.",
        "Review pipeline pressure at the affected zone.",
    ]

    if "both" in t or ("flow" in t and "pressure" in t):
        actions, header = flow_actions + pressure_actions, "Flow + Pressure Anomalies:"
    elif "flow" in t or "leak" in t:
        actions, header = flow_actions, "Flow Anomalies:"
    elif "pressure" in t:
        actions, header = pressure_actions, "Pressure Anomalies:"
    else:
        actions, header = flow_actions + pressure_actions, "General:"

    return header + "\n" + "\n".join(f"  {i+1}. {a}" for i, a in enumerate(actions))


tools = [get_water_system_analysis, get_anomaly_details, get_weather_data, get_corrective_actions]

print("Registered tools:")
for t in tools:
    print(f"  • {t.name}")

# Quick tool test
print()
print(get_weather_data.invoke({"date_str": "2025-10-05"}))
print()
print(get_weather_data.invoke({"date_str": "not-a-date"}))


Registered tools:
  • get_water_system_analysis
  • get_anomaly_details
  • get_weather_data
  • get_corrective_actions

Weather on 2025-10-05 — Ramallah, Palestine:
  Max Temp      : 26.7 °C
  Min Temp      : 17.1 °C
  Precipitation : 0.0 mm
  Note: Weather is additional context only.

[Weather Error] Invalid date 'not-a-date'. Please use YYYY-MM-DD format. Detail: Unknown datetime string format, unable to parse: not-a-date, at position 0


## 6. Build the LangChain Agent


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL   = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")

if not GOOGLE_API_KEY:
    raise EnvironmentError("GOOGLE_API_KEY is not set in .env")

llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=GOOGLE_API_KEY,
    timeout=60,
    max_retries=2,
)

SYSTEM_PROMPT = """
You are an AI assistant for water utility managers.
Your role is to monitor, analyze, and report on water system performance.

Rules:
- Always use the provided tools to retrieve data before answering.
- Never fabricate or estimate numbers not returned by a tool.
- Anomalies are indicators, not confirmed failures. Use language like 'possible leak' or 'may indicate'.
- Estimated anomalous flow volume is a proxy estimate, not a physical leak measurement.
- Weather data is additional context only — do not claim it caused an anomaly.
- If a tool returns an error, report the limitation clearly instead of guessing.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    name="water_utility_agent",
)

print(f"Agent ready. Model: {GEMINI_MODEL}")


Agent ready. Model: gemini-3.6-flash


### Structured Output
Define a Pydantic schema so the Agent returns a structured, validated report.


In [9]:
from pydantic import BaseModel, Field
from typing import List


class KeyMetrics(BaseModel):
    total_demand_m3               : float = Field(description="Total water demand in m3.")
    estimated_anomalous_volume_m3 : float = Field(description="Proxy estimate of anomalous flow volume in m3 (not a physical leak measurement).")
    efficiency_percent            : float = Field(description="System efficiency percentage.")


class WaterSystemReport(BaseModel):
    summary            : str        = Field(description="Natural-language summary of system performance.")
    key_metrics        : KeyMetrics = Field(description="Key performance metrics.")
    anomalies          : List[str]  = Field(description="List of detected anomaly descriptions.")
    corrective_actions : List[str]  = Field(description="List of recommended corrective actions.")
    weather_information: str        = Field(description="Weather context for the queried date.")
    limitations        : List[str]  = Field(description="Known data limitations and uncertainties.")


structured_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    response_format=WaterSystemReport,
    name="water_utility_structured_agent",
)

print("Structured agent ready.")
print("Response format: WaterSystemReport (Pydantic)")


Structured agent ready.
Response format: WaterSystemReport (Pydantic)


## 7. Run the Agent & Generate Report


In [10]:
from langchain_google_genai.chat_models import GoogleRateLimitError

user_query = (
    "Analyze the water system performance, identify the detected anomalies, "
    "check the weather for 2025-10-05, and recommend corrective actions."
)

print("User Query:", user_query)
print("=" * 60)

result = None
try:
    result = structured_agent.invoke({
        "messages": [{"role": "user", "content": user_query}]
    })
    print("Agent completed successfully.")

except GoogleRateLimitError:
    print("[Rate Limit] Daily free-tier quota reached (20 req/day).")
except Exception as e:
    print(f"[Agent Error] {type(e).__name__}: {e}")


User Query: Analyze the water system performance, identify the detected anomalies, check the weather for 2025-10-05, and recommend corrective actions.


Agent completed successfully.


In [11]:
if result is None:
    print("No result to display.")
else:
    # Natural-language report
    print("=" * 60)
    print("NATURAL-LANGUAGE REPORT")
    print("=" * 60)
    last_msg = result["messages"][-1].content
    if isinstance(last_msg, list):
        for block in last_msg:
            if isinstance(block, dict) and "text" in block:
                print(block["text"])
    else:
        print(last_msg)

    # Structured output
    print()
    print("=" * 60)
    print("STRUCTURED OUTPUT (WaterSystemReport)")
    print("=" * 60)
    report = result.get("structured_response")
    if report is None:
        print("Structured response not available.")
    else:
        print(f"Summary : {report.summary}")
        print(f"\nKey Metrics:")
        print(f"  Total Demand                    : {report.key_metrics.total_demand_m3:.2f} m3")
        print(f"  Estimated Anomalous Flow Volume : {report.key_metrics.estimated_anomalous_volume_m3:.2f} m3")
        print(f"  Efficiency                      : {report.key_metrics.efficiency_percent:.2f}%")
        print(f"\nAnomalies:")
        for a in report.anomalies:
            print(f"  • {a}")
        print(f"\nCorrective Actions:")
        for act in report.corrective_actions:
            print(f"  • {act}")
        print(f"\nWeather: {report.weather_information}")
        print(f"\nLimitations:")
        for lim in report.limitations:
            print(f"  • {lim}")


NATURAL-LANGUAGE REPORT
{"summary":"The water system operated at 96.48% efficiency with a total demand of 10946.00 m3. A total of 59 anomalies were detected, comprising 42 flow anomalies and 17 pressure anomalies, with a proxy estimated anomalous volume of 385.30 m3.","key_metrics":{"total_demand_m3":10946.0,"estimated_anomalous_volume_m3":385.3,"efficiency_percent":96.48},"anomalies":["42 flow anomalies detected across METER_001, METER_002, and METER_003, which may indicate a possible pipeline leak, unusual water demand, or meter calibration issues.","17 pressure anomalies detected across METER_001, METER_002, and METER_003, which may indicate a possible pressure drop, pump issue, or valve issue."],"corrective_actions":["Inspect the pipeline segment for possible leaks.","Check and calibrate the meters.","Compare readings with nearby meters.","Check pumps and review operating schedules.","Inspect valves for blockage or partial closure.","Review pipeline pressure at the affected zone."]

## 8. Test by Me
Try asking the AI Agent custom questions to test its reasoning and tool usage.


In [12]:
custom_query_1 = "أعطني تفاصيل عن شواذ الضغط وما هي الإجراءات التي يجب على فريق الصيانة اتخاذها؟"

custom_query_2 = "ما هي حالة الطقس ليوم 2025-10-02 وهل توضح أي مشكلة في تدفق المياه؟"

print("السؤال:", custom_query_1)
print("=" * 60)
try:
    res = structured_agent.invoke({"messages": [{"role": "user", "content": custom_query_1}]})
    report = res.get("structured_response")
    if report:
        print(f"الملخص: {report.summary}\n")
        print("الإجراءات التصحيحية:")
        for act in report.corrective_actions:
            print(f"  • {act}")
    else:
        print(res["messages"][-1].content)
except Exception as e:
    print(f"[خطأ / تجاوز الحد اليومي] {e}")


السؤال: أعطني تفاصيل عن شواذ الضغط وما هي الإجراءات التي يجب على فريق الصيانة اتخاذها؟
الملخص: تم رصد 17 شذوذاً في الضغط من أصل 59 شذوذاً كلياً في النظام عبر العدادات METER_001 و METER_002 و METER_003. قد تشير هذه الشواذ إلى انخفاض محتمل في الضغط، أو مشاكل في المضخات، أو انسداد/خلل في الصمامات.

الإجراءات التصحيحية:
  • فحص المضخات ومراجعة جداول التشغيل.
  • فحص الصمامات للتأكد من عدم وجود انسداد أو إغلاق جزئي.
  • مراجعة ضغط شبكة الأنابيب في المنطقة المتأثرة.
